# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data. These cases cover strictness on output and input, text
forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import json
import math

from schemas.Framework import JSON, Plain, Proxies, Schemas as S
from support import raises, same_graph, text

Item = S.OfObject.Builder().properties(text("s"), text("i", int), text("f", float), text("b", bool),
                                       text("raw", bytes)).create()
Link = S.OfRelation.Builder().links("from", "to").properties(text("w", float)).create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("out").of(Link).me("from"),
                                          lambda r: r.name("in").of(Link).me("to")).update()
Proxies.register("Item", Item)
Proxies.register("Link", Link)
B = Proxies.Builders
FromJSON = JSON.FromJSON(B)

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for value in [math.nan, math.inf, -math.inf]:
    out = JSON.ToJSON(Item, B.Item().f(value).create())
    json.loads(out, parse_constant=lambda c: (_ for _ in ()).throw(AssertionError(c)))
    assert json.loads(out)["objects"]["s0"]["f"] in ("NaN", "Infinity", "-Infinity")
with raises(ValueError):
    JSON.dumps({"x": math.nan})  # plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for bad in ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', '[NaN]',
            '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
            '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', '', '   ', '{"a": 1} {"b": 2}', '{"a": .5}']:
    with raises(ValueError):
        JSON.loads(bad)

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
doc = '{"a": "é"}'
assert JSON.loads(doc) == JSON.loads(doc.encode("utf-8")) == {"a": "é"}
for encoding in ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]:
    assert JSON.loads(doc.encode(encoding)) == {"a": "é"}, encoding
with raises(ValueError, match="BOM"):
    JSON.loads("﻿" + doc)  # a BOM inside a str is an error; in bytes it is accepted
with raises(ValueError):
    JSON.loads(b"\xff\xfe\x00")  # undecodable bytes

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
item = B.Item().raw(b"\x01").s("日本").i(1).create()
compact = JSON.ToJSON(Item, item)
assert compact == '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}'
pretty = JSON.ToJSON(Item, item, indent=2)
assert "\n" in pretty and JSON.loads(pretty) == JSON.loads(compact)
assert JSON.ToJSON.OfObject(Item, item, indent=4) == JSON.dumps(Plain.ToPlain(Item, item), indent=4)

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for value in [0, -1, 2**53 + 1, 2**100, -(2**200)]:
    assert FromJSON(Item, JSON.ToJSON(Item, B.Item().i(value).create())).i == value
for value in [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]:
    back = FromJSON(Item, JSON.ToJSON(Item, B.Item().f(value).create())).f
    assert back == value and math.copysign(1, back) == math.copysign(1, value)
for value in ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\U0001f600", "a\r\nb"]:
    assert FromJSON(Item, JSON.ToJSON(Item, B.Item().s(value).create())).s == value, repr(value)
assert FromJSON(Item, JSON.ToJSON(Item, B.Item().raw(bytes(range(256))).create())).raw == bytes(range(256))

## JSN-06 · A lone surrogate is valid Python JSON text but cannot be encoded to UTF-8 (pinned)

In [ ]:
lone = JSON.ToJSON(Item, B.Item().s("\ud800").create())
with raises(UnicodeEncodeError):
    lone.encode("utf-8")
assert "\ud800" in lone  # callers writing files should use errors="surrogatepass" or reject such strings

## JSN-07 · Natives at the top level

In [ ]:
assert JSON.ToJSON(S.OfNative.Data(int), 5) == "5"
assert JSON.ToJSON(S.OfNative.Data(bytes), b"\xff") == '"/w=="'
assert JSON.ToJSON.OfNative(S.OfNative.Data(float), math.inf) == '"Infinity"'
assert FromJSON(S.OfNative.Data(str), '"x"') == FromJSON.OfNative(S.OfNative.Data(str), '"x"') == "x"
with raises(TypeError):
    FromJSON(S.OfNative.Data(int), "true")
with raises(TypeError):
    FromJSON(S.OfNative.Data(float), "1")  # JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
hub = B.Item().s("hub").out(lambda x: x.to(lambda y: y.s("leaf1")).w(1.0)).out(
    lambda x: x.to(lambda y: y.s("leaf2")).w(math.nan)).create()
graph = Plain.ToPlain.Reachable(Item, hub)
text_ = JSON.ToJSON.Reachable(Item, hub)
assert JSON.loads(text_) == graph
back = FromJSON.Reachable(Item, text_)
assert back is not hub and same_graph(Plain.ToPlain.Reachable(Item, back), graph)
with raises(ValueError, match="unresolved reference"):
    FromJSON(Item, JSON.ToJSON(Item, hub))
with raises(ValueError):
    FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"', 1))